# Agents & Tool Use

Companion notebook for the [Agents & Tool Use lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/05-agents-and-tool-use).

**The idea in one sentence.** A **ReAct** agent interleaves **reasoning** ("Thought:")
and **acting** ("Action: tool[arg]") in a loop — the model decides which tool to call,
observes the result, and continues — with a **bounded step budget** so it can't loop
forever.

The essentials:

- **Thought → Action → Observation** loop: the model reasons, calls a tool, sees the
  result, repeats.
- **Bounded loop:** a `max_steps` budget is the runtime guardrail that guarantees
  termination.

We build a tiny ReAct agent from scratch, **validate that the tools ground the answer and
the loop terminates**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A tiny ReAct agent

An agent calls **tools** and reasons over their results. We build a bounded ReAct loop with a scripted policy (standing in for the LLM) that emits `Action: tool[arg]` lines; our runtime executes the tool and feeds back an `Observation`.

In [ ]:
def calculator(expr):
    return str(eval(expr, {'__builtins__': {}}, {}))

FACTS = {'capital of france': 'Paris', 'population of paris': '2100000'}
def lookup(query):
    return FACTS.get(query.strip().lower(), 'unknown')

TOOLS = {'calculator': calculator, 'lookup': lookup}

# Scripted 'policy' — in a real agent the LLM produces these lines.
SCRIPT = [
    'Thought: I need the capital of France.\nAction: lookup[capital of france]',
    'Thought: Now its population.\nAction: lookup[population of paris]',
    'Thought: Divide by 1000.\nAction: calculator[2100000 / 1000]',
    'Thought: I have the answer.\nAnswer: about 2100',
]

In [ ]:
def run_agent(script, max_steps=6):
    for step in range(max_steps):
        msg = script[step]
        print(msg)
        m = re.search(r'Action:\s*(\w+)\[(.*?)\]', msg)
        if m:
            tool, arg = m.group(1), m.group(2)
            obs = TOOLS[tool](arg)
            print(f'Observation: {obs}\n')
        elif 'Answer:' in msg:
            print('\n[done]')
            return
    print('\n[stopped: step budget exhausted]')

run_agent(SCRIPT)

### Validate: tools return grounded results and the loop is bounded

Two checks. The tools produce *correct*, verifiable results (a calculator computes, a
lookup returns known facts) — grounding the agent. And the `max_steps` budget guarantees
the loop terminates even if the script never emits an answer.

In [ ]:
# tools are correct and grounded
assert TOOLS['calculator']('2 + 2 * 3') == '8', 'calculator computes correctly'
assert TOOLS['lookup']('capital of france') == 'Paris', 'lookup returns known facts'
assert TOOLS['lookup']('unknown thing') == 'unknown', 'lookup fails gracefully'
print('calculator("2 + 2 * 3") =', TOOLS['calculator']('2 + 2 * 3'))
print('lookup("capital of france") =', TOOLS['lookup']('capital of france'))

# the loop is bounded: a script with no 'Answer:' still terminates at max_steps
never_ends = ['Thought: loop\nAction: lookup[capital of france]'] * 10
run_agent(never_ends, max_steps=3)     # must stop, not run forever
print('\n✅ tools ground the agent with verifiable results, and max_steps guarantees termination')

The `max_steps` budget is the **bounded loop** that stops a runaway agent. Note we sandbox `eval` (no builtins) — never feed unvalidated model output to a raw interpreter.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **unsandboxed eval** | `eval()`-ing model output is remote code execution (demo) |
| **no step budget** | agents loop forever without a `max_steps` guardrail |
| **tool errors** | tools fail; the agent must observe and recover, not crash |
| **wrong tool choice** | the model may call the wrong tool; keep the tool set small and clear |
| **compounding errors** | multi-step trajectories compound per-step failure (p^n) |

Demo: a hardened calculator rejects non-arithmetic input instead of executing it.

In [ ]:
# The safety flip-side: this toy's calculator uses eval(). Even with builtins removed,
# eval() on model-proposed strings is dangerous — a real tool must allowlist inputs, never
# eval arbitrary text. We show a hardened calculator that rejects anything non-arithmetic.
import re as _re
def safe_calc(expr):
    if not _re.fullmatch(r'[\d\s.+\-*/()]+', expr):
        return 'Error: rejected non-arithmetic input'
    return str(eval(expr, {'__builtins__': {}}, {}))
print(safe_calc('2 + 2'))
print(safe_calc('__import__("os").system("ls")'))
assert safe_calc('__import__("os").system("ls")').startswith('Error'), 'malicious input must be rejected'
print('\nNever eval() raw model output — allowlist the characters/operations a tool accepts.')

## ✏️ Your turn

Add a `length` tool that returns the number of characters in its argument, and register it in `TOOLS`.

In [ ]:
def length(s):
    # TODO(you): return the character count of s as a string.
    return ''

TOOLS['length'] = length
assert TOOLS['length']('paris') == '5'
print('passed ✓')

<details><summary>Solution</summary>

```python
def length(s):
    return str(len(s))
```

</details>

## Key takeaways

- **ReAct = Thought → Action → Observation** in a loop; the model chooses tools, observes
  results, and continues.
- **Tools ground the agent** with verifiable results (verified) — the fix for
  hallucination.
- **Bound the loop:** a `max_steps` budget guarantees termination (verified) — an agent
  must never loop forever.
- **Sandbox tool inputs:** never `eval()` raw model output; allowlist what a tool accepts
  (demo).